# apertus-eval-prep — paper-matrix **vLLM backend** (Colab ID-5)

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_stability_backend.ipynb)

Runtime → Change runtime type → **T4 GPU**.

This notebook is only the paper-matrix **`backend=vllm`** arm (3 models × 800). HF quant / seed / sampled stay in [`colab_stability.ipynb`](colab_stability.ipynb).

Do **not** Run all. Every session: **cell 1 → 2 → 3 (Drive) → one model sweep**.

Same Drive folder as the HF notebook (`MyDrive/apertus-eval-prep-paper`) so partials and `registry_paper.jsonl` merge.

In [ ]:
# Cell 1 — clone or pull
import os
if os.path.exists("pyproject.toml") and os.path.exists("src/apertus_eval_prep"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"
!pip -q install vllm
!git log -1 --oneline

In [ ]:
# Cell 2 — pull + GPU + vLLM check (rerun every session)
import os, sys
from pathlib import Path

if Path("pyproject.toml").exists() and Path("src/apertus_eval_prep").exists():
    pass
elif Path("apertus-eval-prep/pyproject.toml").exists():
    os.chdir("apertus-eval-prep")
else:
    raise FileNotFoundError("Run cell 1 first (clone).")

!git pull --ff-only
!pip -q install -e ".[gpu,viz]"
!pip -q install vllm
!git log -1 --oneline

_repo_src = str((Path.cwd() / "src").resolve())
if _repo_src not in sys.path:
    sys.path.insert(0, _repo_src)

import torch
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))
import vllm
print("vllm", getattr(vllm, "__version__", "?"))

if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

In [ ]:
# Cell 3 — Drive (same folder as HF paper matrix)
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run the clone/pip cell first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
n_partial = len(list(Path("results/runs").glob("*.partial.jsonl")))
print(f"partial checkpoints on disk: {n_partial}", flush=True)

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]
            i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]
            i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep in-process model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    n_skip = sum(1 for p in planned if p["skipped"])
    print({"n_cells": len(planned), "n_skip": n_skip}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --only-factor backend --out-dir results/runs --registry results/registry_paper.jsonl | head -n 30

## ID-5 sweeps — run one model per session

T4 profile: SmolLM2 + Qwen-3B + Phi only (7B vLLM skipped).

In [ ]:
# ID-5 — SmolLM2 backend=vllm (one 800-item run)
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "backend")

In [ ]:
# ID-5 — Qwen-3B backend=vllm (one 800-item run)
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "backend")

In [ ]:
# ID-5 — Phi-3.5 backend=vllm (one 800-item run)
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "backend")

Unpack the Drive zip into the Mac clone. Commit new `results/runs/*.json` and `results/registry_paper.jsonl`. Do not edit numbers.